In [1]:
import pandas as pd
import driver_number_lookup as dnl

import warnings
warnings.filterwarnings('ignore')

3
max_verstappen
None


# Mapping Kaggle

In [2]:
# Load Kaggle Data 
races = pd.read_csv("../data/kaggle-base-data/races.csv")
results = pd.read_csv("../data/kaggle-base-data/results.csv")
quali = pd.read_csv("../data/kaggle-base-data/qualifying.csv")
drivers = pd.read_csv("../data/kaggle-base-data/drivers.csv") # Needed to map driverRef to your slugs

# Filter all pre 2022
rel_races = races[(races['year'] >= 2022)].copy()

# Filter for Non-Sprint Races
standard_races = rel_races[rel_races['sprint_date'].isna() | (rel_races['sprint_date'] == r'\N')].copy()

# Keep only the columns we need to identify the race
standard_races = standard_races[['raceId', 'year', 'round', 'name', 'circuitId']]


# Merge Results and Quali to get the targets
# Merge Results and Quali to get the targets
race_results = results.merge(standard_races, on='raceId', how='inner')

# Merge with qualifying to get Q3/Q2/Q1 performance
df_spine = race_results.merge(
    quali[['raceId', 'driverId', 'position', 'q1', 'q2', 'q3']], 
    on=['raceId', 'driverId'], 
    how='left',
    suffixes=('_race', '_quali')
)

# 1. Convert Kaggle's existing number column (from results) to an integer
# It often imports as a string with '\N' for missing values
df_spine['number'] = pd.to_numeric(df_spine['number'], errors='coerce')

# 2. Standardize Driver IDs using their Racing Number
def get_canonical_slug_by_number(row):
    year = row['year']
    number = row['number']
    
    if pd.isna(number):
        return None
        
    # Use the helper function you already built in driver_number_lookup.py!
    return dnl.get_driver_name(year, int(number))

df_spine['driver_slug'] = df_spine.apply(get_canonical_slug_by_number, axis=1)

# Drop drivers that aren't in your mapping
df_spine = df_spine.dropna(subset=['driver_slug'])

# Clean and define the target variables
final_spine = pd.DataFrame({
    'season': df_spine['year'],
    'round': df_spine['round'],
    'race_name': df_spine['name'],
    'driver_slug': df_spine['driver_slug'],
    'driver_number': df_spine['number'],
    'team_id': df_spine['constructorId'], 
    
    # Targets for the ML Models
    'target_quali_pos': df_spine['position_quali'],
    'target_race_pos': df_spine['positionOrder'], 
    'target_is_dnf': df_spine['statusId'].apply(lambda x: 1 if x != 1 and x != 11 else 0) 
})

# Sort chronologically
final_spine = final_spine.sort_values(['season', 'round', 'target_race_pos']).reset_index(drop=True)

In [3]:
# ==========================================
# PHASE 2: HISTORICAL FEATURE ENGINEERING
# ==========================================

# 1. Sort by driver and time to ensure sequential rolling windows
final_spine = final_spine.sort_values(by=['driver_slug', 'season', 'round']).reset_index(drop=True)

# 2. Shift the targets by 1 to create "Previous Race" features
# If you don't shift, the model will use the Monza result to predict the Monza result!
final_spine['prev_race_pos'] = final_spine.groupby('driver_slug')['target_race_pos'].shift(1)
final_spine['prev_quali_pos'] = final_spine.groupby('driver_slug')['target_quali_pos'].shift(1)

# 3. Calculate Rolling 3-Race and 5-Race Averages for the Driver
# min_periods=1 ensures we get data even at the start of a season or for rookies
final_spine['driver_form_3_race'] = final_spine.groupby('driver_slug')['prev_race_pos'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
final_spine['driver_form_5_race'] = final_spine.groupby('driver_slug')['prev_race_pos'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)

final_spine['driver_quali_form_3'] = final_spine.groupby('driver_slug')['prev_quali_pos'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# 4. Track Driver Reliability (Rolling DNF Rate)
final_spine['prev_dnf'] = final_spine.groupby('driver_slug')['target_is_dnf'].shift(1)
final_spine['driver_dnf_rate_5_race'] = final_spine.groupby('driver_slug')['prev_dnf'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)

# Sort back to chronological race order for easy viewing
final_spine = final_spine.sort_values(['season', 'round', 'target_race_pos']).reset_index(drop=True)



In [4]:
# ==========================================
# PHASE 3: CONSTRUCTOR FORM & TEAMMATE H2H
# ==========================================

# 1. CONSTRUCTOR FORM (How fast is the car right now?)
# The true pace of a car is best represented by its HIGHEST finishing driver (ignoring the 2nd driver who might have crashed)
team_race_best = final_spine.groupby(['season', 'round', 'team_id'])['target_race_pos'].min().reset_index()
team_race_best.rename(columns={'target_race_pos': 'team_best_finish'}, inplace=True)

# Sort chronologically to do rolling calculations for the team
team_race_best = team_race_best.sort_values(['team_id', 'season', 'round'])

# Shift to prevent data leakage (use previous race's best finish)
team_race_best['prev_team_best_finish'] = team_race_best.groupby('team_id')['team_best_finish'].shift(1)

# Calculate rolling 3-race and 5-race average for the constructor's peak performance
team_race_best['constructor_form_3_race'] = team_race_best.groupby('team_id')['prev_team_best_finish'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Merge constructor form back into the main spine
final_spine = final_spine.merge(
    team_race_best[['season', 'round', 'team_id', 'constructor_form_3_race']], 
    on=['season', 'round', 'team_id'], 
    how='left'
)

# 2. TEAMMATE HEAD-TO-HEAD (Who is dominating the garage?)
# We use the shifted 'prev_race_pos' and 'prev_quali_pos' we created in Phase 2 to avoid data leakage
final_spine['team_rank_prev_race'] = final_spine.groupby(['season', 'round', 'team_id'])['prev_race_pos'].rank(method='min')
final_spine['team_rank_prev_quali'] = final_spine.groupby(['season', 'round', 'team_id'])['prev_quali_pos'].rank(method='min')

# If team_rank == 1.0, they beat their teammate. Create a binary feature (1 if they won, 0 if they lost/tied)
final_spine['beat_teammate_prev_race'] = (final_spine['team_rank_prev_race'] == 1.0).astype(int)
final_spine['beat_teammate_prev_quali'] = (final_spine['team_rank_prev_quali'] == 1.0).astype(int)

# Calculate their rolling 5-race H2H win rate against their teammate
final_spine['h2h_race_win_rate_5'] = final_spine.groupby('driver_slug')['beat_teammate_prev_race'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)
final_spine['h2h_quali_win_rate_5'] = final_spine.groupby('driver_slug')['beat_teammate_prev_quali'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)

# Sort back to chronological race order to keep things clean
final_spine = final_spine.sort_values(['season', 'round', 'target_race_pos']).reset_index(drop=True)

In [5]:
# ==========================================
# PHASE 4: TRACK AFFINITY & CIRCUIT TYPES
# ==========================================
import numpy as np

# 1. Bring `circuitId` into the spine (since we didn't include it in the Phase 1 definition)
# standard_races already has 'year', 'round', and 'circuitId'
circuit_map = standard_races[['year', 'round', 'circuitId']].rename(columns={'year': 'season'})
final_spine = final_spine.merge(circuit_map, on=['season', 'round'], how='left')

# 2. HISTORICAL TRACK AFFINITY
# Sort by driver, circuit, and time to calculate historical average at THIS specific track
final_spine = final_spine.sort_values(by=['driver_slug', 'circuitId', 'season', 'round']).reset_index(drop=True)

# Shift by 1 so we only look at previous years' results at this track, not the current one
final_spine['prev_track_race_pos'] = final_spine.groupby(['driver_slug', 'circuitId'])['target_race_pos'].shift(1)

# Calculate expanding mean (average of all previous appearances at this track)
# min_periods=1 means if they've only raced here once before, it uses that single result
final_spine['driver_track_affinity'] = final_spine.groupby(['driver_slug', 'circuitId'])['prev_track_race_pos'].transform(
    lambda x: x.expanding().mean()
)

# NOTE: Rookies racing at a track for the first time will have NaN. 
# LightGBM handles NaNs natively, but you can also backfill with their general `driver_form_5_race`.
final_spine['driver_track_affinity'] = final_spine['driver_track_affinity'].fillna(final_spine['driver_form_5_race'])


# 3. CIRCUIT TYPE CHARACTERISTICS
# Kaggle circuitIds for modern tracks (2022+). We map them to characteristics.
# 1=Albert Park, 4=Catalunya, 6=Monaco, 9=Silverstone, 14=Monza, 15=Singapore, 22=Suzuka, 73=Baku, etc.
street_circuits = [1, 6, 15, 73, 77, 79, 80] # Melbourne, Monaco, Singapore, Baku, Jeddah, Miami, Vegas
low_downforce_circuits = [14, 13, 73, 77, 80] # Monza, Spa, Baku, Jeddah, Vegas (High speed)

# Create categorical/binary features
final_spine['is_street_circuit'] = final_spine['circuitId'].apply(lambda x: 1 if x in street_circuits else 0)
final_spine['is_low_downforce'] = final_spine['circuitId'].apply(lambda x: 1 if x in low_downforce_circuits else 0)


# 4. DRIVER/TEAM SPECIALIZATION (e.g., "Street Circuit Specialists")
# Let's see how well a driver performs specifically on street circuits
# Sort chronologically across ALL tracks
final_spine = final_spine.sort_values(by=['driver_slug', 'season', 'round']).reset_index(drop=True)

# Mask the previous race position to only show if it was a street circuit
final_spine['prev_street_pos'] = np.where(
    final_spine.groupby('driver_slug')['is_street_circuit'].shift(1) == 1,
    final_spine.groupby('driver_slug')['target_race_pos'].shift(1),
    np.nan
)

# Calculate their rolling average on the last 3 street circuits they raced
final_spine['street_circuit_form'] = final_spine.groupby('driver_slug')['prev_street_pos'].transform(
    lambda x: x.rolling(window=20, min_periods=1).mean() # Window is large to look back for the last few street races
)

# Final sort back to chronological order
final_spine = final_spine.sort_values(['season', 'round', 'target_race_pos']).reset_index(drop=True)

In [6]:
# =========================================================
# PHASE 4.5: INTEGRATE OPENF1 LOCAL TELEMETRY
# =========================================================
import os
import pandas as pd
import numpy as np

def load_local_fp_pace(year, race_folder_name, session_name="FP2"):
    """Loads lap times from local OpenF1 data and calculates baseline pace."""
    base_path = f"../data/openf1-telemetry-data/{year}-{race_folder_name}"
    
    # ⚠️ Check this file name! Adjust to match your actual files (e.g. "FP2_laps.csv")
    file_path = os.path.join(base_path, f"{session_name}.csv")
    
    if not os.path.exists(file_path):
        return None
        
    df_laps = pd.read_csv(file_path)
    df_laps = df_laps.dropna(subset=['lap_duration'])
    
    # 107% rule to filter out slow laps
    absolute_fastest = df_laps['lap_duration'].min()
    cutoff_time = absolute_fastest * 1.07
    representative_laps = df_laps[df_laps['lap_duration'] <= cutoff_time]
    
    driver_pace = representative_laps.groupby('driver_number').agg(
        fp_fastest_lap=('lap_duration', 'min')
    ).reset_index()
    
    # Normalize to Delta (0.000s for the fastest, +X.XXXs for everyone else)
    session_best = driver_pace['fp_fastest_lap'].min()
    driver_pace['fp_pace_delta'] = driver_pace['fp_fastest_lap'] - session_best
    
    driver_pace['driver_slug'] = driver_pace['driver_number'].apply(
        lambda x: dnl.get_driver_name(year, int(x))
    )
    driver_pace['season'] = year
    
    return driver_pace[['season', 'driver_slug', 'fp_pace_delta']]


print("Integrating local Free Practice data into the ML Spine...")
all_fp_data = []

# Loop through races from 2023 onwards
races_to_check = standard_races[standard_races['year'] >= 2023]

for _, row in races_to_check.iterrows():
    year = row['year']
    round_num = row['round']
    
    folder_name = row['name'].replace(" Grand Prix", "")
    # Add any manual mappings here if folder names differ from Kaggle names
    mapping = {"United States": "Austin", "Emilia Romagna": "Imola"}
    folder_name = mapping.get(folder_name, folder_name)
    
    pace_df = load_local_fp_pace(year, folder_name)
    
    if pace_df is not None:
        pace_df['round'] = round_num
        all_fp_data.append(pace_df)

# Merge into final_spine
if all_fp_data:
    full_telemetry = pd.concat(all_fp_data, ignore_index=True)
    final_spine = final_spine.merge(full_telemetry, on=['season', 'round', 'driver_slug'], how='left')
    print("✅ Free Practice pace successfully added to final_spine!")
else:
    print("⚠️ No local telemetry found. Creating a blank column to prevent crashes.")
    final_spine['fp_pace_delta'] = np.nan

Integrating local Free Practice data into the ML Spine...
✅ Free Practice pace successfully added to final_spine!


In [7]:
# ==========================================
# PHASE 5: THE BASELINE RANKING MODEL
# ==========================================
import lightgbm as lgb
import numpy as np

# 1. Define our Feature Set (X)
features = [
    'fp_pace_delta',
    'driver_form_3_race', 'driver_form_5_race', 'driver_quali_form_3',
    'driver_dnf_rate_5_race', 'constructor_form_3_race', 
    'beat_teammate_prev_race', 'beat_teammate_prev_quali',
    'h2h_race_win_rate_5', 'h2h_quali_win_rate_5', 
    'driver_track_affinity', 'is_street_circuit', 
    'is_low_downforce', 'street_circuit_form'
]

# Fill NaNs
final_spine['fp_pace_delta'] = final_spine['fp_pace_delta'].fillna(1.5)
final_spine[features] = final_spine[features].fillna(-999)

# 2. Walk-Forward Split (Predicting 2026 - Round 13)
target_season = 2026
target_round = 16

train_df = final_spine[
    (final_spine['season'] < target_season) | 
    ((final_spine['season'] == target_season) & (final_spine['round'] < target_round))
].copy()

test_df = final_spine[
    (final_spine['season'] == target_season) & 
    (final_spine['round'] == target_round)
].copy()

# 3. Prepare LightGBM Ranker Data
train_df = train_df.sort_values(['season', 'round'])
test_df = test_df.sort_values(['season', 'round'])

X_train = train_df[features]
# FLIP THE TARGET: Max grid size + 1 (e.g., 22) minus the position. 
# Now P1 = 21 (Highest Relevance), P20 = 2 (Lower Relevance)
y_train = 22 - train_df['target_quali_pos'] 
q_train = train_df.groupby(['season', 'round']).size().to_numpy()

X_test = test_df[features]
y_test = 22 - test_df['target_quali_pos']
q_test = test_df.groupby(['season', 'round']).size().to_numpy()

# 4. Train the Model
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)

ranker.fit(
    X_train, y_train, 
    group=q_train,
    eval_set=[(X_test, y_test)],
    eval_group=[q_test],
)

# 5. Predict and Evaluate 2026 Round 13
test_df['predicted_quali_score'] = ranker.predict(X_test)

# Because a higher predicted score now correctly means P1, we rank descending.
test_df['predicted_quali_rank'] = test_df['predicted_quali_score'].rank(ascending=False)

print(f"\n--- {target_season} Round {target_round} Qualifying Prediction (Kaggle Baseline) ---")
results_view = test_df[['driver_slug', 'target_quali_pos', 'predicted_quali_rank']].sort_values('predicted_quali_rank')
print(results_view.to_string(index=False))

print(f"Predicting: {test_df['season'].iloc[0]} {test_df['race_name'].iloc[0]} (Round {test_df['round'].iloc[0]})")


mae = np.abs(results_view['target_quali_pos'] - results_view['predicted_quali_rank']).mean()
print(f"\nBaseline Mean Absolute Error: {mae:.2f} grid positions")


# 6. Feature Importance Evaluation
# Extracting importance by 'gain' (how much this feature improved the model's accuracy)
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance_Gain': ranker.booster_.feature_importance(importance_type='gain')
})

# Sort from most important to least important
importance_df = importance_df.sort_values(by='Importance_Gain', ascending=False).reset_index(drop=True)

# Normalize the importance to percentages so it's easier to read
total_gain = importance_df['Importance_Gain'].sum()
importance_df['Relative_Importance_%'] = (importance_df['Importance_Gain'] / total_gain) * 100

print("\n--- Model Feature Importance (What drives the predictions?) ---")
# Print formatting to show 1 decimal place for percentages
print(importance_df[['Feature', 'Relative_Importance_%']].to_string(
    index=False, 
    formatters={'Relative_Importance_%': '{:.1f}%'.format}
))


--- 2026 Round 16 Qualifying Prediction (Kaggle Baseline) ---
          driver_slug  target_quali_pos  predicted_quali_rank
       george_russell               8.0                   1.0
andrea_kimi_antonelli               4.0                   2.0
      charles_leclerc               5.0                   3.0
         isack_hadjar               3.0                   4.0
        oscar_piastri               7.0                   5.0
       max_verstappen               1.0                   6.0
       lewis_hamilton               2.0                   7.0
         lando_norris               6.0                   8.0
          liam_lawson              11.0                   9.0
         pierre_gasly               9.0                  10.0
       arvid_lindblad              16.0                  11.0
       oliver_bearman              18.0                  12.0
    gabriel_bortoleto              10.0                  13.0
     franco_colapinto              15.0                  14.0
      n

In [8]:
import numpy as np

def build_upcoming_race(df_spine, races_df, target_season, target_round):
    """Builds the feature grid for a future race without needing results.csv"""
    
    # 1. Get the circuit ID for the upcoming race from the races schedule
    upcoming_race_info = races_df[
        (races_df['year'] == target_season) & 
        (races_df['round'] == target_round)
    ]
    
    if upcoming_race_info.empty:
        raise ValueError(f"Round {target_round} not found in races.csv schedule.")
        
    circuit_id = upcoming_race_info['circuitId'].values[0]
    race_name = upcoming_race_info['name'].values[0]
    
    # 2. Get the active drivers and teams from the most recent race we have data for
    latest_available_round = df_spine[df_spine['season'] == target_season]['round'].max()
    last_race = df_spine[
        (df_spine['season'] == target_season) & 
        (df_spine['round'] == latest_available_round)
    ]
    
    # Initialize the future grid
    future_grid = last_race[['driver_slug', 'team_id']].drop_duplicates().copy()
    future_grid['season'] = target_season
    future_grid['round'] = target_round
    
    # Map track characteristics
    street_circuits = [1, 6, 15, 73, 77, 79, 80]
    low_downforce_circuits = [14, 13, 73, 77, 80]
    future_grid['is_street_circuit'] = 1 if circuit_id in street_circuits else 0
    future_grid['is_low_downforce'] = 1 if circuit_id in low_downforce_circuits else 0
    future_grid['street_circuit_form'] = last_race['street_circuit_form'].values # Carry forward

    # 3. Calculate latest form for each driver
    latest_stats = []
    for driver in future_grid['driver_slug']:
        driver_hist = df_spine[df_spine['driver_slug'] == driver].sort_values(['season', 'round'])
        
        # Get actual finishes
        recent_finishes = driver_hist['target_race_pos'].tail(5).tolist()
        recent_quali = driver_hist['target_quali_pos'].tail(3).tolist()
        recent_dnfs = driver_hist['target_is_dnf'].tail(5).tolist()
        
        # Track affinity
        track_hist = driver_hist[driver_hist['circuitId'] == circuit_id]
        affinity = track_hist['target_race_pos'].mean() if not track_hist.empty else np.mean(recent_finishes)
        
        latest_stats.append({
            'driver_slug': driver,
            'driver_form_3_race': np.mean(recent_finishes[-3:]) if recent_finishes else -999,
            'driver_form_5_race': np.mean(recent_finishes) if recent_finishes else -999,
            'driver_quali_form_3': np.mean(recent_quali) if recent_quali else -999,
            'driver_dnf_rate_5_race': np.mean(recent_dnfs) if recent_dnfs else 0,
            'driver_track_affinity': affinity,
            'h2h_race_win_rate_5': driver_hist['h2h_race_win_rate_5'].iloc[-1] if not driver_hist.empty else 0.5,
            'h2h_quali_win_rate_5': driver_hist['h2h_quali_win_rate_5'].iloc[-1] if not driver_hist.empty else 0.5,
        })
        
    stats_df = pd.DataFrame(latest_stats)
    future_grid = future_grid.merge(stats_df, on='driver_slug', how='left')
    
    # 4. Constructor Form
    team_stats = []
    for team in future_grid['team_id'].unique():
        team_hist = df_spine[df_spine['team_id'] == team].sort_values(['season', 'round'])
        team_stats.append({
            'team_id': team,
            'constructor_form_3_race': team_hist['constructor_form_3_race'].iloc[-1] if not team_hist.empty else -999
        })
        
    future_grid = future_grid.merge(pd.DataFrame(team_stats), on='team_id', how='left')
    
    # 5. Teammate H2H (Last race context)
    future_grid['beat_teammate_prev_race'] = last_race['beat_teammate_prev_race'].values
    future_grid['beat_teammate_prev_quali'] = last_race['beat_teammate_prev_quali'].values

    return future_grid, race_name

# =========================================================
# RUN PREDICTION FOR FUTURE RACE (Round 18)
# =========================================================

# Ensure standard_races is passed here to get the circuit ID
X_future, race_name = build_upcoming_race(final_spine, standard_races, target_season=2026, target_round=18)

X_future['fp_pace_delta'] = 1.5

# Fill any NaNs safely
X_future[features] = X_future[features].fillna(-999)

# Predict using the model you trained earlier
X_future['predicted_quali_score'] = ranker.predict(X_future[features])
X_future['predicted_quali_rank'] = X_future['predicted_quali_score'].rank(ascending=False)

print(f"\n--- 2026 {race_name} (Round 18) - FUTURE QUALIFYING PREDICTION ---")
print(X_future[['driver_slug', 'predicted_quali_rank']].sort_values('predicted_quali_rank').to_string(index=False))


--- 2026 United States Grand Prix (Round 18) - FUTURE QUALIFYING PREDICTION ---
          driver_slug  predicted_quali_rank
      charles_leclerc                   1.0
andrea_kimi_antonelli                   2.0
         lando_norris                   3.0
       lewis_hamilton                   4.0
         isack_hadjar                   5.0
       max_verstappen                   6.0
       george_russell                   7.0
        oscar_piastri                   8.0
         lance_stroll                   9.0
    gabriel_bortoleto                  10.0
       oliver_bearman                  11.0
         carlos_sainz                  12.0
          liam_lawson                  13.0
     franco_colapinto                  14.0
         pierre_gasly                  15.0
      nico_hulkenberg                  16.0
         esteban_ocon                  17.0
       arvid_lindblad                  18.0
         sergio_perez                  19.0
      fernando_alonso                  

In [9]:
# ==========================================
# PHASE 6: THE RACE RANKING MODEL
# ==========================================

# 1. Update Features to include the Qualifying Grid
race_features = [
    'target_quali_pos', 
    'fp_pace_delta',
    'driver_form_3_race', 'driver_form_5_race', 'driver_quali_form_3',
    'driver_dnf_rate_5_race', 'constructor_form_3_race', 
    'beat_teammate_prev_race', 'beat_teammate_prev_quali',
    'h2h_race_win_rate_5', 'h2h_quali_win_rate_5', 
    'driver_track_affinity', 'is_street_circuit', 
    'is_low_downforce', 'street_circuit_form'
]

final_spine[race_features] = final_spine[race_features].fillna(-999)

# 2. Extract X and flip the new Y target (Race Position)
X_train_race = train_df[race_features]
# FLIP THE TARGET: P1 becomes 21 (Highest Relevance), P20 becomes 2
y_train_race = 22 - train_df['target_race_pos'] 

X_test_race = test_df[race_features]
y_test_race = 22 - test_df['target_race_pos']

# 3. Train the Race Model
race_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)

race_ranker.fit(
    X_train_race, y_train_race, 
    group=q_train,
    eval_set=[(X_test_race, y_test_race)],
    eval_group=[q_test],
)

# 4. Predict and Evaluate the Race
test_df['predicted_race_score'] = race_ranker.predict(X_test_race)
test_df['predicted_race_rank'] = test_df['predicted_race_score'].rank(ascending=False)

print(f"\n--- {target_season} Round {target_round} RACE PREDICTION ---")
race_results_view = test_df[['driver_slug', 'target_quali_pos', 'target_race_pos', 'predicted_race_rank']].sort_values('predicted_race_rank')
print(race_results_view.to_string(index=False))

race_mae = np.abs(race_results_view['target_race_pos'] - race_results_view['predicted_race_rank']).mean()
print(f"\nRace Baseline MAE: {race_mae:.2f} positions")

# 5. Race Feature Importance
race_importance = pd.DataFrame({
    'Feature': race_features,
    'Gain': race_ranker.booster_.feature_importance(importance_type='gain')
})
race_importance['Relative_%'] = (race_importance['Gain'] / race_importance['Gain'].sum()) * 100
race_importance = race_importance.sort_values(by='Gain', ascending=False).reset_index(drop=True)

print("\n--- Race Model Feature Importance ---")
print(race_importance[['Feature', 'Relative_%']].to_string(
    index=False, formatters={'Relative_%': '{:.1f}%'.format}
))


--- 2026 Round 16 RACE PREDICTION ---
          driver_slug  target_quali_pos  target_race_pos  predicted_race_rank
       max_verstappen               1.0                1                  1.0
       lewis_hamilton               2.0                3                  2.0
         isack_hadjar               3.0                5                  3.0
andrea_kimi_antonelli               4.0                2                  4.0
       george_russell               8.0               20                  5.0
      charles_leclerc               5.0                4                  6.0
         lando_norris               6.0                9                  7.0
         pierre_gasly               9.0               16                  8.0
        oscar_piastri               7.0                6                  9.0
          liam_lawson              11.0                7                 10.0
    gabriel_bortoleto              10.0               18                 11.0
       arvid_lindblad    

In [10]:
from lightgbm import LGBMClassifier

# ==========================================
# PHASE 8: RACE DNF PROBABILITY CLASSIFIER
# ==========================================

# 1. We use the exact same training features as the Race model
# But our target is now the binary 'target_is_dnf' column
y_train_dnf = train_df['target_is_dnf']
y_test_dnf = test_df['target_is_dnf']

# 2. Train a Classifier instead of a Ranker
dnf_classifier = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

dnf_classifier.fit(X_train_race, y_train_dnf)

# Evaluate the Classifier on our test set (e.g., Round 16)
# predict_proba returns an array of [Prob_0, Prob_1]. We want Prob_1 (chance of DNF)
test_df['predicted_dnf_prob'] = dnf_classifier.predict_proba(X_test_race)[:, 1]

print("\n--- DNF Probability Test (Historical Race) ---")
dnf_view = test_df[['driver_slug', 'target_is_dnf', 'predicted_dnf_prob']].sort_values('predicted_dnf_prob', ascending=False)
# Format as percentage for readability
dnf_view['predicted_dnf_prob'] = (dnf_view['predicted_dnf_prob'] * 100).round(1).astype(str) + '%'
print(dnf_view.to_string(index=False))


--- DNF Probability Test (Historical Race) ---
          driver_slug  target_is_dnf predicted_dnf_prob
         sergio_perez              0              94.4%
      valtteri_bottas              1              93.5%
         lance_stroll              0              48.5%
      fernando_alonso              0              40.6%
           alex_albon              1              28.6%
      nico_hulkenberg              0              25.9%
       george_russell              1              21.8%
         esteban_ocon              0              21.8%
       oliver_bearman              0              21.8%
    gabriel_bortoleto              0              18.6%
         carlos_sainz              0              16.3%
     franco_colapinto              0              14.5%
       arvid_lindblad              0              13.4%
         pierre_gasly              0              12.9%
       lewis_hamilton              0               7.1%
          liam_lawson              0               6.6%


In [11]:
# =========================================================
# PHASE 7: FUTURE RACE INFERENCE (QUALI & RACE & DNF)
# =========================================================

def predict_future_weekend(df_spine, races_df, target_season, target_round, ranker_quali, ranker_race, quali_features, race_features, live_fp_data=None):
    """Builds the grid, predicts Quali, feeds Quali into Race, predicts Race & DNF."""
    
    # 1. Get Race Info
    upcoming_race = races_df[(races_df['year'] == target_season) & (races_df['round'] == target_round)]
    if upcoming_race.empty:
        raise ValueError(f"Round {target_round} not found.")
        
    circuit_id = upcoming_race['circuitId'].values[0]
    race_name = upcoming_race['name'].values[0]
    race_date = upcoming_race['date'].values[0] if 'date' in upcoming_race.columns else "Unknown Date"
    
    # 2. Build the baseline grid
    latest_round = df_spine[df_spine['season'] == target_season]['round'].max()
    last_race = df_spine[(df_spine['season'] == target_season) & (df_spine['round'] == latest_round)]
    
    future_grid = last_race[['driver_slug', 'team_id']].drop_duplicates().copy()
    future_grid['season'] = target_season
    future_grid['round'] = target_round
    
    # Track mapping
    street_circuits = [1, 6, 15, 73, 77, 79, 80]
    low_downforce_circuits = [14, 13, 73, 77, 80]
    future_grid['is_street_circuit'] = 1 if circuit_id in street_circuits else 0
    future_grid['is_low_downforce'] = 1 if circuit_id in low_downforce_circuits else 0
    future_grid['street_circuit_form'] = last_race['street_circuit_form'].values

    # 3. Calculate Driver Momentum
    latest_stats = []
    for driver in future_grid['driver_slug']:
        driver_hist = df_spine[df_spine['driver_slug'] == driver].sort_values(['season', 'round'])
        recent_finishes = driver_hist['target_race_pos'].tail(5).tolist()
        recent_quali = driver_hist['target_quali_pos'].tail(3).tolist()
        recent_dnfs = driver_hist['target_is_dnf'].tail(5).tolist()
        track_hist = driver_hist[driver_hist['circuitId'] == circuit_id]
        affinity = track_hist['target_race_pos'].mean() if not track_hist.empty else np.mean(recent_finishes)
        
        latest_stats.append({
            'driver_slug': driver,
            'driver_form_3_race': np.mean(recent_finishes[-3:]) if recent_finishes else -999,
            'driver_form_5_race': np.mean(recent_finishes) if recent_finishes else -999,
            'driver_quali_form_3': np.mean(recent_quali) if recent_quali else -999,
            'driver_dnf_rate_5_race': np.mean(recent_dnfs) if recent_dnfs else 0,
            'driver_track_affinity': affinity,
            'h2h_race_win_rate_5': driver_hist['h2h_race_win_rate_5'].iloc[-1] if not driver_hist.empty else 0.5,
            'h2h_quali_win_rate_5': driver_hist['h2h_quali_win_rate_5'].iloc[-1] if not driver_hist.empty else 0.5,
        })
    future_grid = future_grid.merge(pd.DataFrame(latest_stats), on='driver_slug', how='left')
    
    # 4. Constructor Momentum & H2H
    team_stats = []
    for team in future_grid['team_id'].unique():
        team_hist = df_spine[df_spine['team_id'] == team].sort_values(['season', 'round'])
        team_stats.append({'team_id': team, 'constructor_form_3_race': team_hist['constructor_form_3_race'].iloc[-1] if not team_hist.empty else -999})
    future_grid = future_grid.merge(pd.DataFrame(team_stats), on='team_id', how='left')
    future_grid['beat_teammate_prev_race'] = last_race['beat_teammate_prev_race'].values
    future_grid['beat_teammate_prev_quali'] = last_race['beat_teammate_prev_quali'].values

    # Clean missing data
    if live_fp_data is not None:
        # If it's Friday night and we have live FP pace, merge it in!
        future_grid = future_grid.merge(live_fp_data[['driver_slug', 'fp_pace_delta']], on='driver_slug', how='left')
        
        # If a driver didn't run in FP (e.g. car broke down), default them to 1.5s off pace
        future_grid['fp_pace_delta'] = future_grid['fp_pace_delta'].fillna(1.5)
    else:
        # If it's Wednesday and cars haven't hit the track yet, assume everyone is 1.5s off pace.
        # The model will naturally fall back to relying heavily on historical form.
        future_grid['fp_pace_delta'] = 1.5
        
    # Clean missing data
    future_grid[quali_features] = future_grid[quali_features].fillna(-999)

    # ---------------------------------------------------------
    # STEP A: PREDICT QUALIFYING
    # ---------------------------------------------------------
    future_grid['predicted_quali_score'] = ranker_quali.predict(future_grid[quali_features])
    future_grid['predicted_quali_rank'] = future_grid['predicted_quali_score'].rank(ascending=False)
    
    # ---------------------------------------------------------
    # STEP B: PREDICT RACE (Using predicted quali as a feature)
    # ---------------------------------------------------------
    future_grid['target_quali_pos'] = future_grid['predicted_quali_rank']
    future_grid['predicted_race_score'] = ranker_race.predict(future_grid[race_features])
    future_grid['predicted_race_rank'] = future_grid['predicted_race_score'].rank(ascending=False)

    # ---------------------------------------------------------
    # STEP C: PREDICT RACE DNF PROBABILITY
    # ---------------------------------------------------------
    # Now that 'target_quali_pos' and all momentum features exist, we can predict DNF!
    future_grid['predicted_dnf_prob'] = dnf_classifier.predict_proba(future_grid[race_features])[:, 1]
    
    return future_grid, race_name, race_date

# --- RUN IT ---
# 1. Pull the live data for the upcoming weekend (Using the function from Phase 11)
# live_weekend_pace = get_fp_pace(year=2026, country_name="United States", session_name="Practice 2")

# 2. Pass it into the predictor
future_results, r_name, r_date = predict_future_weekend(
    df_spine=final_spine, 
    races_df=races, 
    target_season=2026, 
    target_round=18,
    ranker_quali=ranker, 
    ranker_race=race_ranker,
    quali_features=features, 
    race_features=race_features,
    live_fp_data=None # <--- Pass `live_weekend_pace` here if it's Friday night!
)

print(f"\n=======================================================")
print(f" 🏁 FUTURE PREDICTION: {target_season} {r_name} (Round 18)")
print(f" 📅 Date: {r_date}")
print(f"=======================================================\n")

# Format DNF Prob as a readable percentage
future_results['DNF_Risk'] = (future_results['predicted_dnf_prob'] * 100).round(1).astype(str) + '%'

# Display the full weekend prediction sorted by Race Finish
final_view = future_results[['driver_slug', 'predicted_quali_rank', 'predicted_race_rank', 'DNF_Risk']].sort_values('predicted_race_rank')
final_view.columns = ['Driver', 'Predicted Quali', 'Predicted Race Finish', 'DNF Risk']
print(final_view.to_string(index=False))


 🏁 FUTURE PREDICTION: 2026 United States Grand Prix (Round 18)
 📅 Date: 2026-10-25

               Driver  Predicted Quali  Predicted Race Finish DNF Risk
      charles_leclerc              1.0                    1.0     4.6%
andrea_kimi_antonelli              2.0                    2.0     3.5%
         isack_hadjar              5.0                    3.0     3.4%
         lando_norris              3.0                    4.0     4.7%
       george_russell              7.0                    5.0     4.6%
       max_verstappen              6.0                    6.0    10.6%
       lewis_hamilton              4.0                    7.0     9.8%
         lance_stroll              9.0                    8.0    11.9%
       oliver_bearman             11.0                    9.0     9.3%
        oscar_piastri              8.0                   10.0     5.6%
    gabriel_bortoleto             10.0                   11.0    10.6%
          liam_lawson             13.0                   12.0  

In [12]:
# =========================================================
# PHASE 9: F1 FANTASY SCORING ENGINE (P10, P50, P90)
# =========================================================

def calculate_fantasy_percentiles(row, mae=3.5):
    """Calculates the P10, P50, and P90 fantasy points for a driver."""
    base_q = row['predicted_quali_rank']
    base_r = row['predicted_race_rank']
    dnf_prob = row['predicted_dnf_prob']
    
    # Helper function for F1 Fantasy Points math
    def get_points(q_pos, r_pos, beat_teammate, is_dnf):
        pts = 0
        # 1. Quali Points (Top 10)
        if q_pos <= 10: pts += (11 - q_pos)
        
        # 2. Race Finish Points
        finish_pts = {1:25, 2:18, 3:15, 4:12, 5:10, 6:8, 7:6, 8:4, 9:2, 10:1}
        pts += finish_pts.get(int(r_pos), 0)
        
        # 3. Positions Gained/Lost
        pts += max(min(q_pos - r_pos, 10), -10)
        
        # 4. Teammate & DNF
        if beat_teammate: pts += 10  # +5 Quali, +5 Race
        if is_dnf: pts -= 20
        return pts

    # --- P50: The Standard Expectation ---
    # Finishes exactly as predicted. Expected value for teammate is +5 pts (50/50). 
    # DNF penalty is scaled mathematically by probability.
    p50 = get_points(base_q, base_r, beat_teammate=False, is_dnf=False) + 5 - (dnf_prob * 20)
    
    # --- P90: The Upside Ceiling ---
    # Driver out-qualifies and out-races prediction by MAE bounds, beats teammate, no DNF.
    p90_q = max(1, base_q - mae)
    p90_r = max(1, base_r - mae)
    p90 = get_points(p90_q, p90_r, beat_teammate=True, is_dnf=False)
    
    # --- P10: The Downside Floor ---
    # Driver underperforms by MAE, loses to teammate. 
    # If their DNF risk is >10%, the bottom 10th percentile outcome mathematically includes a crash.
    p10_q = min(20, base_q + mae)
    p10_r = min(20, base_r + mae)
    is_p10_dnf = True if dnf_prob > 0.10 else False
    p10 = get_points(p10_q, p10_r, beat_teammate=False, is_dnf=is_p10_dnf)
    
    return pd.Series([round(p10, 1), round(p50, 1), round(p90, 1)])

# Apply the percentiles to our future predictions
future_results[['P10_Pts', 'P50_Pts', 'P90_Pts']] = future_results.apply(calculate_fantasy_percentiles, axis=1)

print("\n=======================================================")
print(" 🏆 F1 FANTASY PROJECTIONS (P10 - P50 - P90) ")
print("=======================================================\n")

# Display sorted by the most likely outcome (P50)
fantasy_view = future_results[['driver_slug', 'predicted_quali_rank', 'predicted_race_rank', 'DNF_Risk', 'P10_Pts', 'P50_Pts', 'P90_Pts']].sort_values('P50_Pts', ascending=False)
fantasy_view.columns = ['Driver', 'Pred Quali', 'Pred Race', 'DNF Risk', 'P10 (Floor)', 'P50 (Expected)', 'P90 (Ceiling)']
print(fantasy_view.to_string(index=False))


 🏆 F1 FANTASY PROJECTIONS (P10 - P50 - P90) 

               Driver  Pred Quali  Pred Race DNF Risk  P10 (Floor)  P50 (Expected)  P90 (Ceiling)
      charles_leclerc         1.0        1.0     4.6%         18.5            39.1           45.0
andrea_kimi_antonelli         2.0        2.0     3.5%         15.5            31.3           45.0
         isack_hadjar         5.0        3.0     3.4%         12.5            27.3           45.0
         lando_norris         3.0        4.0     4.7%          9.5            23.1           45.0
       george_russell         7.0        5.0     4.6%          6.0            20.1           44.5
       max_verstappen         6.0        6.0    10.6%        -16.5            15.9           36.5
       lewis_hamilton         4.0        7.0     9.8%          1.5            13.0           32.5
         lance_stroll         9.0        8.0    11.9%        -19.0             9.6           28.5
       oliver_bearman        11.0        9.0     9.3%          2.0     

In [13]:
# =========================================================
# PHASE 10: MLFLOW & DAGSHUB LOGGING (Models + Feature Store)
# =========================================================
import mlflow
import dagshub
import json

# 1. Connect to your DagsHub Repository 
# (Make sure these match your exact DagsHub URL: dagshub.com/SamuelFredricBerg/dt911)
dagshub.init(repo_owner="SamuelFredricBerg", repo_name="4dt911", mlflow=True)

# Set the experiment name
mlflow.set_experiment("F1_Fantasy_Baseline")

# 2. Build the "Feature Store" for the Live Website
# We take the future_results dataframe we generated in Phase 7 and convert it to a dictionary.
# We set driver_slug as the index so your website can do O(1) lookups (e.g. data["lando_norris"])
feature_store_df = future_results.set_index('driver_slug')

# Select only the raw input features (so the website can pass them into the model)
all_needed_features = list(set(features + race_features))
feature_store_dict = feature_store_df[all_needed_features].to_dict(orient='index')

# Save it locally as a JSON file
with open("upcoming_race_features.json", "w") as f:
    json.dump(feature_store_dict, f, indent=4)

# 3. Start the tracking run and upload everything to the cloud
with mlflow.start_run(run_name="Kaggle_Historical_Baseline_v1"):
    
    # A. Log Configurations & Parameters
    mlflow.log_params({
        "quali_features": features,
        "race_features": race_features,
        "n_estimators": 100,
        "ranker_learning_rate": 0.1,
        "dnf_learning_rate": 0.05,
        "eval_season": target_season,
        "eval_round": target_round
    })
    
    # B. Log Evaluation Metrics
    mlflow.log_metrics({
        "quali_mae": mae,
        "race_mae": race_mae
    })
    
    safe_types = [
        "lightgbm.sklearn.LGBMRanker", 
        "lightgbm.sklearn.LGBMClassifier"
    ]
    
    mlflow.lightgbm.log_model(ranker, "quali_ranker_model", skops_trusted_types=safe_types)
    mlflow.lightgbm.log_model(race_ranker, "race_ranker_model", skops_trusted_types=safe_types)
    mlflow.lightgbm.log_model(dnf_classifier, "dnf_classifier_model", skops_trusted_types=safe_types)
    
    # D. Log the Feature Store (The "Baseline Data" for the website sliders)
    mlflow.log_artifact("upcoming_race_features.json", artifact_path="feature_store")
    
    print("✅ Models, configs, metrics, AND Feature Store successfully logged to DagsHub!")

Accessing as Soppa22

Initialized MLflow to track repo "SamuelFredricBerg/4dt911"

Repository SamuelFredricBerg/4dt911 initialized!

2026/11/02 17:02:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/02 17:02:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/02 17:03:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Models, configs, metrics, AND Feature Store successfully logged to DagsHub!
🏃 View run Kaggle_Historical_Baseline_v1 at: https://dagshub.com/SamuelFredricBerg/4dt911.mlflow/#/experiments/0/runs/833c363d84014310a13e160dab037b7c
🧪 View experiment at: https://dagshub.com/SamuelFredricBerg/4dt911.mlflow/#/experiments/0


In [14]:
def get_driver_fantasy_prediction(driver_slug, season, round_num, df_spine, races_df):
    """
    Given a specific driver and future round, returns their F1 Fantasy projections.
    """
    # 1. Run the full weekend simulation for the requested round
    future_grid, r_name, r_date = predict_future_weekend(
        df_spine=df_spine, 
        races_df=races_df, 
        target_season=season, 
        target_round=round_num,
        ranker_quali=ranker, 
        ranker_race=race_ranker,
        quali_features=features, 
        race_features=race_features
    )
    
    # 2. Apply the Fantasy Scoring Math
    future_grid[['P10_Pts', 'P50_Pts', 'P90_Pts']] = future_grid.apply(calculate_fantasy_percentiles, axis=1)
    
    # 3. Extract just the requested driver
    driver_data = future_grid[future_grid['driver_slug'] == driver_slug]
    
    if driver_data.empty:
        return {"error": f"Driver '{driver_slug}' not found in active grid for Round {round_num}."}
        
    driver_row = driver_data.iloc[0]
    
    # Return a clean JSON/Dict format perfect for a web API
    return {
        "race_name": r_name,
        "date": r_date,
        "driver": driver_slug,
        "predictions": {
            "qualifying_position": int(driver_row['predicted_quali_rank']),
            "race_position": int(driver_row['predicted_race_rank']),
            "dnf_probability": round(driver_row['predicted_dnf_prob'] * 100, 1)
        },
        "fantasy_points": {
            "floor_p10": driver_row['P10_Pts'],
            "expected_p50": driver_row['P50_Pts'],
            "ceiling_p90": driver_row['P90_Pts']
        }
    }

# --- Example Usage for your Web Backend ---
prediction_json = get_driver_fantasy_prediction("lando_norris", 2026, 18, final_spine, races)
print(prediction_json)

{'race_name': 'United States Grand Prix', 'date': '2026-10-25', 'driver': 'lando_norris', 'predictions': {'qualifying_position': 3, 'race_position': 4, 'dnf_probability': np.float64(4.7)}, 'fantasy_points': {'floor_p10': np.float64(9.5), 'expected_p50': np.float64(23.1), 'ceiling_p90': np.float64(45.0)}}
